<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day03-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 3 — In-class discussion problem (2 of 3)

Work this out **by hand in your group first** — then run the code cell to check your answer before presenting.

## Official name or accession number — which do you search with?

A colleague hands you the RefSeq mRNA accession `NM_000546` (no version, no gene name) and says "this is the gene I'm working on."

1. Predict: if you search UniProt's website for the bare string `NM_000546`, do you expect to find the right protein entry, find nothing, or find several entries?
2. Now predict what happens if you instead search UniProt for the official gene symbol together with a "reviewed only" filter.
3. Which of the two searches would you trust to hand a student who has never seen this gene before, and why?

In [1]:
import requests

def search(query, size=10):
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/search",
        params={"query": query, "size": size, "fields": "accession,id,reviewed"},
        timeout=10,
    )
    r.raise_for_status()
    return r.json()["results"]

by_accession = search("NM_000546")
by_symbol = search("gene:TP53 AND organism_id:9606 AND reviewed:true")

print(f"Searching by bare RefSeq accession -> {len(by_accession)} hit(s):")
for hit in by_accession:
    print("  ", hit["primaryAccession"], hit["uniProtkbId"], hit["entryType"])

print(f"\nSearching by official symbol + reviewed:true -> {len(by_symbol)} hit(s):")
for hit in by_symbol:
    print("  ", hit["primaryAccession"], hit["uniProtkbId"], hit["entryType"])

Searching by bare RefSeq accession -> 3 hit(s):
   P04637 P53_HUMAN UniProtKB reviewed (Swiss-Prot)
   K7PPA8 K7PPA8_HUMAN UniProtKB unreviewed (TrEMBL)
   Q53GA5 Q53GA5_HUMAN UniProtKB unreviewed (TrEMBL)

Searching by official symbol + reviewed:true -> 1 hit(s):
   P04637 P53_HUMAN UniProtKB reviewed (Swiss-Prot)


**Discussion point:** the bare-accession search actually *does* find the right entry (`P04637`, `P53_HUMAN`) — UniProt indexes cross-referenced RefSeq IDs too — but it comes back alongside unreviewed TrEMBL entries that turn out to be *identical-sequence* records filed under different accessions, not new information. The symbol+`reviewed:true` search instead goes straight to the single canonical, curated entry.

Neither search is "wrong," but they answer different questions: an accession search tells you "which records mention this exact identifier," which is contaminated by database redundancy; a symbol search on the official name is what actually gets a newcomer to the one entry everyone else in the field means by "TP53."